In [1]:
from dotenv import load_dotenv
load_dotenv('../env')

True

# Imports

In [2]:
import gc
import os
import torch
import wandb
from tqdm.auto import tqdm
import jsonlines

from datasets import load_dataset, Dataset
from typing import Union, Dict, Any

from transformers import AutoTokenizer, TrainingArguments
from unsloth import FastLanguageModel

from trl import ORPOConfig, ORPOTrainer

# Define Variables

In [3]:
## Models
# MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'
# MODEL_ID = 'unsloth/mistral-7b-instruct-v0.2-bnb-4bit'
# MODEL_ID = 'unsloth/gemma-2b-it-bnb-4bit'
# MODEL_ID = 'unsloth/gemma-2b-bnb-4bit'
# MODEL_ID = 'unsloth/gemma-7b-it-bnb-4bit'
# MODEL_ID = 'unsloth/llama-3-8b-bnb-4bit'
# MODEL_ID = 'google/gemma-1.1-7b-it'
# MODEL_ID = 'microsoft/Phi-3-mini-128k-instruct'
# MODEL_ID = 'unsloth/Phi-3-mini-4k-instruct-bnb-4bit'
# MODEL_ID = 'unsloth/gemma-2b'
MODEL_ID = 'unsloth/gemma-7b'

## MODEL TRAINING VARIABLE
MAX_LENGTH = 32_768 // 4
BATCH_SIZE = 1
LOAD_IN_4BIT = True
DTYPE = torch.bfloat16  # None for auto-detection

## PROMPT VARIABLES
SYSTEM_PROMPT = False
# SYSTEM_PROMPT = True
USE_CHAT_TEMPLATE = False

## LORA SETTINGS
LORA_R = 128
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_BIAS = 'none'
USE_GRADIENT_CHECKPOINTING = True  # 'unsloth' # True or 'unsloth' for very long context
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

## ORPO Config
LR=8e-6
LR_SCHEDULER_TYPE='cosine'
BETA=0.1
EPOCHS=1
MAX_STEPS=500
WARMUP_STEPS=10
LOGGING_STEPS=1
OUTPUT_DIR=os.path.join('Training_Outputs', f"ORPO-Unsloth-{MODEL_ID.split('/')[-1].replace('.', '_')}")
OPTIMIZER="paged_adamw_8bit"

## WANDB SETUP
wandb.login(key=os.getenv('WANDB_API_KEY'))
os.environ["WANDB_PROJECT"]="ORPO-Training"
os.environ['WANDB_WATCH']='false'
os.environ["WANDB_LOG_MODEL"]='false'
RUN_NAME = f"ORPO-Unsloth-{MODEL_ID.split('/')[-1].replace('.', '_')}-1Epoch"
# RUN_NAME = f"VRAM-Test-With-Unsloth-gemma-2b-ORPO"

## PATHS
# DATASET_FPATH = 'ORPO_Training_Data.parquet'
# DATASET_FPATH = 'ORPO_Training_Data_Gemma.parquet'
# DATASET_FPATH = 'ORPO_Training_Data_Llama3.parquet'
# DATASET_FPATH = 'ORPO_Training_Data_Unsloth_Gemma_2b.parquet'
DATASET_FPATH = 'ORPO_Training_Data_Unsloth_Gemma_7b.parquet'
SAVE_FPATH = f"Finetuned_Models/{RUN_NAME}"


wandb: Currently logged in as: hrushikesh. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/jupyter/.netrc


In [4]:
MAX_LENGTH

8192

# Model Setup

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    use_cache=not USE_GRADIENT_CHECKPOINTING, # if gradient_checkpointing else True
    max_seq_length=MAX_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    device_map={"": 0}
)

Unsloth: You passed in `unsloth/gemma-7b` and `load_in_4bit = True`.
We shall load `unsloth/gemma-7b-bnb-4bit` for 4x faster loading.
/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


==((====))==  Unsloth: Fast Gemma patching release 2024.4
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.151 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 8.0. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. Xformers = 0.0.26.post1. FA = True.
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
Gemma's activation function should be approximate GeLU and not exact GeLU.
Changing the activation function to `gelu_pytorch_tanh`.if you want to use the legacy `gelu`, edit the `model.config` to set `hidden_activation=gelu`   instead of `hidden_act`. See https://github.com/huggingface/transformers/pull/29402 for more details.


In [6]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.model_max_length = MAX_LENGTH
# tokenizer.padding_side = 'right'

In [7]:
if tokenizer.chat_template is None:
    llama_tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B-Instruct')
    tokenizer.chat_template = llama_tokenizer.chat_template
print(tokenizer.chat_template)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>

'+ message['content'] | trim + '<|eot_id|>' %}{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>

' }}{% endif %}


In [8]:
# Add LORA Adapters
model = FastLanguageModel.get_peft_model(
    model=model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias=LORA_BIAS,
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    max_seq_length=MAX_LENGTH,
)

Unsloth 2024.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Data Prep

In [9]:
def get_messages_in_format(sys_prompt:str, user_prompt:str, completion:str=None, use_system:bool=False) -> list:

    if use_system:
        message = [
            {
                'role': 'system',
                'content': sys_prompt
            },
            {
                'role': 'user',
                'content': user_prompt
            }
        ]
    else:
        message = [
            {
                'role': 'user',
                'content': sys_prompt + '\n\n' + user_prompt
            }
        ]
    
    if completion is not None:
        message.append(
            {
                'role': 'assistant',
                'content': completion
            }
        )
    
    
    return message

In [10]:
## Setup Data
dataset = load_dataset("parquet", data_files=DATASET_FPATH)['train']

# Setup Chat Template
def format_chat_template(current_row):
    if USE_CHAT_TEMPLATE:
        # Must add EOS_TOKEN
        current_row['chosen'] = current_row['chosen'][-1]['content'] + tokenizer.eos_token # tokenizer.apply_chat_template(current_row['chosen'], tokenize=False)
        current_row['rejected'] = current_row['rejected'][-1]['content'] + tokenizer.eos_token # tokenizer.apply_chat_template(current_row['rejected'], tokenize=False)

        if SYSTEM_PROMPT:
            current_row['prompt'] = tokenizer.apply_chat_template(get_messages_in_format(sys_prompt=current_row['QA_System_Prompt'], user_prompt=current_row['QA_User_Prompt'], use_system=True), tokenize=False, add_generation_prompt=True)
        else:
            current_row['prompt'] = tokenizer.apply_chat_template(current_row['prompt'], tokenize=False, add_generation_prompt=True)
    else:
        current_row['chosen'] = current_row['chosen'] + tokenizer.eos_token # tokenizer.apply_chat_template(current_row['chosen'], tokenize=False)
        current_row['rejected'] = current_row['rejected'] + tokenizer.eos_token # tokenizer.apply_chat_template(current_row['rejected'], tokenize=False)
    
    # if tokenizer.bos_token not in current_row['prompt']:
    #     current_row['prompt'] = tokenizer.bos_token + current_row['prompt']
    
    return current_row

dataset = dataset.map(
    format_chat_template,
    num_proc=os.cpu_count(),
    # batched=True,
)

train_dataset = dataset
# train_dataset = dataset.train_test_split(shuffle=True, train_size=0.1053)['train']  # 1k
# train_dataset = dataset.train_test_split(shuffle=True, train_size=0.4211)['train']  # 4k
# train_dataset = dataset.train_test_split(shuffle=True, train_size=0.2106)['train']  # 2k
train_dataset

Dataset({
    features: ['Custom_ID', 'QA_System_Prompt', 'QA_User_Prompt', 'chosen', 'rejected', 'prompt'],
    num_rows: 9453
})

In [11]:
print(train_dataset['prompt'][0])

You are a Q&A bot.
Follow all the given instructions when answering the questions.

<Instruction>
- Look for the right context from the given article.
- Explain the context in terms of the question
- Answer the given question
- Finally enclosed the yes/no response XML tags, like <FINAL_ANSWER>yes/no</FINAL_ANSWER>.
- If no relevant context is found, return no as answer
</Instruction>

<Article>
FROM THE ALCONA COUNTY SHERIFF’S OFFICE:

On 01/29/2024 Matthew William Busch, 33 from Bay City Michigan was arraigned in the 81st District Court-Alcona on a charge of Assault w/Intent to Commit Murder and Possession of a Firearm While Intoxicated causing a serious impairment. Both charges are a felony with a maximum of life in prison. He is being held without bond.

***STORY BELOW ORIGINALLY POSTED ON SUNDAY, JANUARY 28.

Suspect arrested following shooting at Glennie Tavern

One man is in custody and another is in a downstate hospital following a shooting at Glennie Tavern Friday night.

Accor

In [12]:
print(train_dataset['chosen'][0])

The article mentions that Matthew William Busch was arraigned in the 81st District Court-Alcona on charges of Assault with Intent to Commit Murder and Possession of a Firearm While Intoxicated causing a serious impairment. This indicates that the 81st District Court is involved in the legal proceedings related to the suspect in the shooting incident at Glennie Tavern.

<FINAL_ANSWER>Yes</FINAL_ANSWER>
</Answer><eos>


# Train the Model

In [13]:
# Enable Reward Modelling Stats
from unsloth import PatchDPOTrainer
PatchDPOTrainer()

In [14]:
orpo_args = ORPOConfig(
    learning_rate=LR,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_LENGTH,
    beta=BETA,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2 * BATCH_SIZE,
    num_train_epochs=EPOCHS,
    # max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    report_to='wandb',
    run_name=RUN_NAME,
    optim=OPTIMIZER,
    remove_unused_columns=False,
    truncation_mode="keep_end",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    dataset_num_proc=os.cpu_count(),
    include_tokens_per_second=True,
)


In [15]:
# Trainer
trainer = ORPOTrainer(
    model=model,
    args=orpo_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

In [16]:
# Show Memory Stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU - {gpu_stats.name}. Max memory - {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU - NVIDIA A100-SXM4-80GB. Max memory - 79.151 GB.
7.133 GB of memory reserved.


In [17]:
# trainer_stats = trainer.train()
trainer_stats = trainer.train(
    resume_from_checkpoint=True
)

Cannot get num_tokens from dataloader
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 9,453 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 1 | Gradient Accumulation steps = 2
\        /    Total batch size = 2 | Total steps = 4,726
 "-____-"     Number of trainable parameters = 400,031,744


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / rejected,logps / chosen,logits / rejected,logits / chosen
3501,1.308500,-0.017692,-0.718279,1.000000,0.700587,-7.182791,-0.176921,158.660065,146.502380
3502,0.831000,-0.008013,-0.628970,1.000000,0.620957,-6.289697,-0.080129,150.207977,123.850510
3503,0.861100,-0.026153,-0.631561,1.000000,0.605408,-6.315612,-0.261534,130.771866,139.691254
3504,0.848400,-0.025593,-0.621395,1.000000,0.595801,-6.213946,-0.255933,139.535828,97.557541
3505,0.477800,-0.009588,-0.646103,1.000000,0.636515,-6.461031,-0.095876,114.701370,81.865265
3506,0.838100,-0.025390,-0.623462,1.000000,0.598072,-6.234621,-0.253902,158.244415,137.454453
3507,1.120400,-0.012935,-0.699668,1.000000,0.686733,-6.996678,-0.129349,173.542572,168.508362
3508,1.149800,-0.002498,-0.727162,1.000000,0.724664,-7.271616,-0.024976,152.094116,136.143295
3509,1.223000,-0.013345,-0.618004,1.000000,0.604658,-6.180038,-0.133455,154.308182,137.277451
3510,0.736400,-0.015456,-0.740857,1.000000,0.725402,-7.408572,-0.154557,145.498138,138.736221


/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


# Final Memory and time stats

In [18]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB")
print(f"Peak reserved memory % of max memory = {used_percentage}%")
print(f"Peak reserved memory for training of max memory = {lora_percentage}%")

2218.2208 seconds used for training.
36.97 minutes used for training.
Peak reserved memory = 67.025 GB
Peak reserved memory for training = 59.892 GB
Peak reserved memory % of max memory = 84.68%
Peak reserved memory for training of max memory = 75.668%


# Inference (Testing)

In [19]:
FastLanguageModel.for_inference(model)

In [20]:
import pandas as pd
df = pd.read_parquet("SFT_Training_Data.parquet")

In [21]:
import time
start = time.time()
idx = 126
device = "cuda"

if USE_CHAT_TEMPLATE:
    if SYSTEM_PROMPT:
        text = tokenizer.apply_chat_template([
                    {
                        'role': 'system',
                        'content': df['System_Prompt'][idx]
                    },
                    {
                        'role': 'user',
                        'content': df['User_Prompt'][idx]
                    }
                ], tokenize=False, add_generation_prompt=True)
    else:
        text = tokenizer.apply_chat_template([
                    {
                        'role': 'user',
                        'content': df['System_Prompt'][idx] + '\n' + df['User_Prompt'][idx]
                    }
                ], tokenize=False, add_generation_prompt=True)
else:
    text = tokenizer.bos_token + df['System_Prompt'][idx].replace('<Instruction>', '\n<Instruction>') + '\n' + df['User_Prompt'][idx].replace('Question:', '<Question>').replace('article>', 'Article>') +'\n</Question>\n\n<Answer>'

inputs = tokenizer(text, return_tensors="pt").to(device)
# print(text)
print('Its Here')
outputs = trainer.model.generate(**inputs,temperature = 0.4,do_sample=True, max_new_tokens=512)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))
end = time.time()
total = end-start
print(total,'secs')

Its Here
<bos><bos>You are a Q&A bot. You should answer questions based on the given article as article only.
Follow all the given instructions when answering the questions.

<Instruction>
- Look for the right context from the given article.
- Explain the context in terms of the question
- Answer the given question
- Finally enclosed the yes/no/not_enough_info response XML tags, like <FINAL_ANSWER>yes/no/not_enough_info</FINAL_ANSWER>.
- If the answer is not present in the given article, the final answer should be <FINAL_ANSWER>not_enough_info</FINAL_ANSWER>
</Instruction>

<Article>
CALUMET TOWNSHIP — The Houghton County Sheriff’s Office reported that a deputy to was dispatched to an area along the Calumet Waterworks Road in response to a major gas leak.

Sgt. Keith Raffaelli said that a deputy according to the call, a buried natural gas line was reported to have surfaced and started spewing gas into the air.
</Article>

<Question>
According to the given article, can we say that the `

In [22]:
import time
start = time.time()
idx = 256
device = "cuda"

if USE_CHAT_TEMPLATE:
    if SYSTEM_PROMPT:
        text = tokenizer.apply_chat_template([
                    {
                        'role': 'system',
                        'content': df['System_Prompt'][idx]
                    },
                    {
                        'role': 'user',
                        'content': df['User_Prompt'][idx]
                    }
                ], tokenize=False, add_generation_prompt=True)
    else:
        text = tokenizer.apply_chat_template([
                    {
                        'role': 'user',
                        'content': df['System_Prompt'][idx] + '\n' + df['User_Prompt'][idx]
                    }
                ], tokenize=False, add_generation_prompt=True)
else:
    text = tokenizer.bos_token + df['System_Prompt'][idx].replace('<Instruction>', '\n<Instruction>') + '\n' + df['User_Prompt'][idx].replace('Question:', '<Question>').replace('article>', 'Article>') +'\n</Question>\n\n<Answer>'

inputs = tokenizer(text, return_tensors="pt").to(device)
# print(text)
print('Its Here')
outputs = trainer.model.generate(**inputs,temperature = 0.4,do_sample=True, max_new_tokens=512)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))
end = time.time()
total = end-start
print(total,'secs')

Its Here
<bos><bos>You are a Q&A bot.
Follow all the given instructions when answering the questions.

<Instruction>
- Look for the right context from the given article.
- Explain the context in terms of the question
- Answer the given question
- Finally enclosed the yes/no response XML tags, like <FINAL_ANSWER>yes/no</FINAL_ANSWER>.
- If no relevant context is found, return no as answer
</Instruction>

<Article>
Bradley E. Keith, 43, of El Dorado Springs, Jimi C. Cole, 44, of Stockton, and Brandon Choate, 44, of Lamar have all been charged with tampering with physical evidence in a felony and abandonment of a corpse following their arrests January 17. They are all being held in the Cedar County jail. According to court records, additional charges may be pending. Both Choate and Cole are being held on $200,000 bonds. Keith is being held with no bond.
</Article>

<Question>
Is the `CITY OF EL DORADO SPRINGS` mentioned in any context or in relation to any event or individual, in the give

# Save Model

In [23]:
# model.save_pretrained(SAVE_FPATH + '-4ksteps')
model.save_pretrained(SAVE_FPATH)
tokenizer.save_pretrained(SAVE_FPATH + '-tokenizer')

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('Finetuned_Models/ORPO-Unsloth-gemma-7b-1Epoch-tokenizer/tokenizer_config.json',
 'Finetuned_Models/ORPO-Unsloth-gemma-7b-1Epoch-tokenizer/special_tokens_map.json',
 'Finetuned_Models/ORPO-Unsloth-gemma-7b-1Epoch-tokenizer/tokenizer.model',
 'Finetuned_Models/ORPO-Unsloth-gemma-7b-1Epoch-tokenizer/added_tokens.json',
 'Finetuned_Models/ORPO-Unsloth-gemma-7b-1Epoch-tokenizer/tokenizer.json')

In [24]:
print("Done")

Done


# Loading Saved Model

In [25]:
SAVE_FPATH

'Finetuned_Models/ORPO-Unsloth-gemma-7b-1Epoch'

In [26]:
from unsloth import FastLanguageModel
model, _ = FastLanguageModel.from_pretrained(
    model_name = SAVE_FPATH, # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = MAX_LENGTH // 2,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference


==((====))==  Unsloth: Fast Gemma patching release 2024.4
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.151 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 8.0. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. Xformers = 0.0.26.post1. FA = True.
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


In [27]:
# # SYSTEM_PROMPT = False
tokenizer = AutoTokenizer.from_pretrained(SAVE_FPATH + '-tokenizer')
# tokenizer.chat_template

In [28]:
import time
start = time.time()
idx = 126
device = "cuda"

if USE_CHAT_TEMPLATE:
    if SYSTEM_PROMPT:
        text = tokenizer.apply_chat_template([
                    {
                        'role': 'system',
                        'content': df['System_Prompt'][idx]
                    },
                    {
                        'role': 'user',
                        'content': df['User_Prompt'][idx]
                    }
                ], tokenize=False, add_generation_prompt=True)
    else:
        text = tokenizer.apply_chat_template([
                    {
                        'role': 'user',
                        'content': df['System_Prompt'][idx] + '\n' + df['User_Prompt'][idx]
                    }
                ], tokenize=False, add_generation_prompt=True)
else:
    text = tokenizer.bos_token + df['System_Prompt'][idx].replace('<Instruction>', '\n<Instruction>') + '\n' + df['User_Prompt'][idx].replace('Question:', '<Question>').replace('article>', 'Article>') +'\n</Question>\n\n<Answer>'

inputs = tokenizer(text, return_tensors="pt").to(device)
# print(text)
print('Its Here')
# outputs = model.generate(**inputs,temperature = 0.4, do_sample=True, max_new_tokens=512)
outputs = model.generate(**inputs,temperature = 0, max_new_tokens=512)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))
end = time.time()
total = end-start
print(total,'secs')

Its Here
<bos><bos>You are a Q&A bot. You should answer questions based on the given article as article only.
Follow all the given instructions when answering the questions.

<Instruction>
- Look for the right context from the given article.
- Explain the context in terms of the question
- Answer the given question
- Finally enclosed the yes/no/not_enough_info response XML tags, like <FINAL_ANSWER>yes/no/not_enough_info</FINAL_ANSWER>.
- If the answer is not present in the given article, the final answer should be <FINAL_ANSWER>not_enough_info</FINAL_ANSWER>
</Instruction>

<Article>
CALUMET TOWNSHIP — The Houghton County Sheriff’s Office reported that a deputy to was dispatched to an area along the Calumet Waterworks Road in response to a major gas leak.

Sgt. Keith Raffaelli said that a deputy according to the call, a buried natural gas line was reported to have surfaced and started spewing gas into the air.
</Article>

<Question>
According to the given article, can we say that the `

In [29]:
import time
start = time.time()
idx = 256
device = "cuda"

if USE_CHAT_TEMPLATE:
    if SYSTEM_PROMPT:
        text = tokenizer.apply_chat_template([
                    {
                        'role': 'system',
                        'content': df['System_Prompt'][idx]
                    },
                    {
                        'role': 'user',
                        'content': df['User_Prompt'][idx]
                    }
                ], tokenize=False, add_generation_prompt=True)
    else:
        text = tokenizer.apply_chat_template([
                    {
                        'role': 'user',
                        'content': df['System_Prompt'][idx] + '\n' + df['User_Prompt'][idx]
                    }
                ], tokenize=False, add_generation_prompt=True)
else:
    text = tokenizer.bos_token + df['System_Prompt'][idx].replace('<Instruction>', '\n<Instruction>') + '\n' + df['User_Prompt'][idx].replace('Question:', '<Question>').replace('article>', 'Article>') +'\n</Question>\n\n<Answer>'
    # text = df['System_Prompt'][idx].replace('<Instruction>', '\n<Instruction>') + '\n' + df['User_Prompt'][idx].replace('Question:', '<Question>').replace('article>', 'Article>') +'\n</Question>\n\n<Answer>'

inputs = tokenizer(text, return_tensors="pt").to(device)
# print(text)
print('Its Here')
outputs = model.generate(**inputs,temperature = 0.4,do_sample=True, max_new_tokens=512)
# outputs = model.generate(**inputs,temperature = 0.4, max_new_tokens=512)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))
end = time.time()
total = end-start
print(total,'secs')

Its Here
<bos><bos>You are a Q&A bot.
Follow all the given instructions when answering the questions.

<Instruction>
- Look for the right context from the given article.
- Explain the context in terms of the question
- Answer the given question
- Finally enclosed the yes/no response XML tags, like <FINAL_ANSWER>yes/no</FINAL_ANSWER>.
- If no relevant context is found, return no as answer
</Instruction>

<Article>
Bradley E. Keith, 43, of El Dorado Springs, Jimi C. Cole, 44, of Stockton, and Brandon Choate, 44, of Lamar have all been charged with tampering with physical evidence in a felony and abandonment of a corpse following their arrests January 17. They are all being held in the Cedar County jail. According to court records, additional charges may be pending. Both Choate and Cole are being held on $200,000 bonds. Keith is being held with no bond.
</Article>

<Question>
Is the `CITY OF EL DORADO SPRINGS` mentioned in any context or in relation to any event or individual, in the give

In [ ]:
model.save_pretrained_merged(f'{SAVE_FPATH}-vllm', tokenizer, save_method='merged_16bit')

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 132.51 out of 167.06 RAM for saving.


100%|██████████| 28/28 [00:00<00:00, 59.09it/s]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving model... This might take 5 minutes for Llama-7b...
Done.


In [ ]:
# model.save_pretrained_merged('Finetuned_Models/ORPO-Unsloth-Mistral-7b-Instruct-lora', tokenizer, save_method='lora')